In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Kişiselleştirilmiş Tavsiye Sistemi

Amaç: Bir kullanıcının izleme geçmişine ve beğenilerine dayanarak, mevcut film kataloğundan kişiselleştirilmiş bir film önermek.

In [ ]:
# Gerekli kütüphaneler: pip install sentence-transformers scikit-learn numpy
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- BİLGİ KAYNAĞI ---
KNOWLEDGE_BASE = """
Kullanıcı Profili: Ayşe Yılmaz.
Beğendiği Türler: Bilim Kurgu, Aksiyon.
İzlediği ve Beğendiği Filmler: 'Dune', 'Blade Runner 2049'.
İzlediği ve Beğenmediği Filmler: 'Aşk Tesadüfleri Sever'.
Anahtar Kelimeler: fütüristik, teknoloji, macera.

Film Kataloğu:
Film: 'Matrix'. Tür: Bilim Kurgu, Aksiyon. Konu: Gerçekliğin bir simülasyon olduğunu keşfeden bir hacker'ın mücadelesi.
Film: 'Not Defteri'. Tür: Romantik, Dram. Konu: İki gencin yıllara yayılan aşk hikayesi.
Film: 'Başlangıç (Inception)'. Tür: Bilim Kurgu, Aksiyon, Gerilim. Konu: İnsanların rüyalarına girerek fikir çalan bir grup uzmanın hikayesi.
"""

# --- MODÜLER FONKSİYONLAR ---
def chunk_text(text: str) -> list[str]:
    # Bu senaryoda her bir profil/film ayrı bir chunk
    return [p.strip() for p in text.split('\n\n') if p.strip()]

def get_embeddings(chunks, model):
    return model.encode(chunks)

def find_most_similar_chunks(query, embeddings, chunks, model, top_k=2): # Profil ve bir film bulması için k=2
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_k_indices = np.argsort(similarities)[-top_k:][::-1]
    return [chunks[i] for i in top_k_indices]

def generate_answer(query, context):
    prompt = f"Kullanıcı ve Katalog Bilgisi: {context}\n\nSoru: {query}\n\nCevap:"
    print("--- LLM'e Gönderilen Nihai Prompt ---\n", prompt)
    
    # Bu senaryoda Mock LLM biraz daha akıllı davranmalı
    user_profile = context.split("Film Kataloğu:")[0]
    mock_response = f"'Başlangıç (Inception)' filmini önerebilirim. Çünkü sizin gibi Bilim Kurgu ve Aksiyon türlerini seven, 'Dune' gibi filmleri izlemiş birisi için rüyalar ve teknoloji teması ilgi çekici olabilir."
    return mock_response

# --- ANA PROGRAM AKIŞI ---
if __name__ == "__main__":
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    user_query = "Beğendiğim türlere ve izlediklerime bakarak bana bir film önerir misin?"
    
    print("--- Kişiselleştirilmiş Tavsiye RAG Demosu ---")
    print(f"Kullanıcı Sorusu: '{user_query}'\n")

    chunks = chunk_text(KNOWLEDGE_BASE)
    print(f"--- 1. Parçalama Sonucu: {len(chunks)} adet profil/katalog bilgisi bulundu. ---\n")
    
    document_embeddings = get_embeddings(chunks, embedding_model)
    
    context_chunks = find_most_similar_chunks(user_query, document_embeddings, chunks, embedding_model)
    context_for_llm = "\n".join(context_chunks)
    print("--- 2. Arama Sonucu: En Alakalı Profil ve Film Bilgileri Bulundu ---\n", context_for_llm, "\n")
    
    final_answer = generate_answer(user_query, context_for_llm)
    print("\n--- 3. Nihai Cevap Üretildi ---")
    print(final_answer)